In [2]:
#!pip install ortools
import numpy as np
import pandas as pd
import urllib.request
import time

from ortools.linear_solver import pywraplp
from IPython.display import display

In [3]:
# Escolher instancia. Pode ser de 1 a 20
instancia = 9

# Tolerância de convergência da bisseção (em itens/corredor)
TOLERANCIA = 0.01

# Limite de iterações da bisseção (segurança)
MAX_ITER = 30

# Escreve o arquivo LP a cada iteração (útil para debug, deixar False em produção)
writelp = False

## Leitura dos Dados

In [4]:
url = f"https://raw.githubusercontent.com/mercadolibre/challenge-sbpo-2025/master/datasets/a/instance_{instancia:04}.txt"

lista_maior = []
with urllib.request.urlopen(url) as f:
    texto = f.read().decode('utf-8')
    for line in texto.split('\n'):
        if line:
            lista_maior.append(list(map(int, line.split())))

In [5]:
n_o = lista_maior[0][0]  # numero de pedidos
n_i = lista_maior[0][1]  # numero de itens
n_a = lista_maior[0][2]  # numero de corredores

print(f'numero de pedidos:   {n_o}')
print(f'numero de itens:     {n_i}')
print(f'numero de corredores:{n_a}')

numero de pedidos:   70
numero de itens:     222
numero de corredores:304


In [6]:
# Pedidos
pedidos_dict = {}
for u in range(0, n_o):
    pedidos_dict[u] = {p: 0 for p in range(0, n_i)}

for i in range(1, n_o + 1):
    nitens = lista_maior[i][0]
    pedido = i - 1
    for j in range(1, nitens + 1):
        item = lista_maior[i][2 * j - 1]
        qtd_item = lista_maior[i][2 * j]
        pedidos_dict[pedido][item] = qtd_item

df_ped = pd.DataFrame.from_dict(pedidos_dict, orient='index')

In [7]:
# Corredores
corredores_dict = {}
for u in range(0, n_a):
    corredores_dict[u] = {p: 0 for p in range(0, n_i)}

for i in range(n_o + 1, n_o + n_a + 1):
    nitens = lista_maior[i][0]
    corredor = i - n_o - 1
    for j in range(1, nitens + 1):
        item = lista_maior[i][2 * j - 1]
        qtd_item = lista_maior[i][2 * j]
        corredores_dict[corredor][item] = qtd_item

df_corr = pd.DataFrame.from_dict(corredores_dict, orient='index')

# Bounds
LB, UB = lista_maior[n_o + n_a + 1][0], lista_maior[n_o + n_a + 1][1]
print(f'LB = {LB} | UB = {UB}')

LB = 52 | UB = 153


## Construção dos Conjuntos

In [8]:
I = list(range(n_i))   # itens
A = list(range(n_a))   # corredores
O = list(range(n_o))   # pedidos

# Total de unidades por pedido
cont_O = {o: sum(pedidos_dict[o][i] for i in I) for o in O}

# Itens presentes em cada pedido (com quantidade > 0)
Io = {o: [i for i, v in pedidos_dict[o].items() if v > 0] for o in O}

# Capacidade total de cada corredor (soma de todos os itens)
Uai_lim = {a: sum(corredores_dict[a][i] for i in I) for a in A}

## Funções Auxiliares

In [9]:
def tot_pedidos(Transportes):
    """Soma total de unidades transportadas na solução."""
    return sum(Transportes.values())

def metrica(Transportes, Corredores_Utilizados):
    """Objetivo real do desafio: itens coletados / corredores visitados."""
    return tot_pedidos(Transportes) / len(Corredores_Utilizados)

def extrai_solucao(Yo, Ya, X):
    """Extrai pedidos, corredores e transportes a partir das variáveis do solver."""
    Pedidos_Escolhidos   = [o for o in O if Yo[o].solution_value() > 0.5]
    Corredores_Utilizados = [a for a in A if Ya[a].solution_value() > 0.5]
    Transportes = {u: X[u].solution_value() for u in X if X[u].solution_value() > 0}
    return Pedidos_Escolhidos, Corredores_Utilizados, Transportes

def calcula_sobras(Transportes):
    """Verifica que nenhum corredor foi usado além da sua capacidade.
    Retorna a sobra total (deve ser >= 0 para solução viável)."""
    df_t = pd.DataFrame(
        [(u[0], u[1], u[2], Transportes[u]) for u in Transportes],
        columns=['Item', 'Pedido', 'Corredor', 'Quantidade']
    )
    df_t['corr_item'] = df_t['Item'].astype(str) + '_' + df_t['Corredor'].astype(str)
    uso = df_t.pivot_table(index='corr_item', values='Quantidade', aggfunc='sum').reset_index()

    df_c = df_corr.melt(ignore_index=False).reset_index()
    df_c.columns = ['corredor', 'item', 'valor']
    df_c['corr_item'] = df_c['item'].astype(str) + '_' + df_c['corredor'].astype(str)

    uso = uso.merge(df_c, on='corr_item', how='left')
    uso['sobra'] = uso['valor'] - uso['Quantidade']
    return uso['sobra'].sum()

## Modelo Parametrizado

A função abaixo recebe um valor `p` e resolve o MILP:

$$\max \sum_{o \in O} \text{cont\_O}[o] \cdot Y_o \;-\; p \cdot \sum_{a \in A} Y_a$$

O valor ótimo desta função, chamado $g(p)$, é:
- $g(p) > 0$ → existe solução com métrica $> p$ (bisseção sobe o limite inferior)
- $g(p) \leq 0$ → não existe solução com métrica $> p$ (bisseção abaixa o limite superior)

Pelo **Teorema de Dinkelbach**, a raiz de $g(p) = 0$ é exatamente o valor ótimo do problema fracionário original.

In [10]:
def resolve_parametrico(p):
    """
    Resolve o MILP parametrizado para um dado p.

    Retorna:
        g_p   : valor objetivo do MILP (g(p) no sentido de Dinkelbach)
        Yo    : dict de variáveis de pedido
        Ya    : dict de variáveis de corredor
        X     : dict de variáveis de transporte
        status: status do solver
    """
    solver = pywraplp.Solver.CreateSolver('CBC')

    # ── Variáveis ──────────────────────────────────────────────────────────────

    # x[i,o,a]: unidades do item i do pedido o coletadas no corredor a
    X = {}
    for o in O:
        for i in Io[o]:          # só itens que o pedido realmente precisa
            for a in A:
                X[(i, o, a)] = solver.IntVar(0, UB, f'x_{i}_{o}_{a}')

    # Yo[o]: 1 se o pedido o está na wave
    Yo = {o: solver.BoolVar(f'Yo_{o}') for o in O}

    # Ya[a]: 1 se o corredor a é visitado
    Ya = {a: solver.BoolVar(f'Ya_{a}') for a in A}

    # ── Restrições ─────────────────────────────────────────────────────────────

    # (R1) Tamanho da wave entre LB e UB
    solver.Add(sum(cont_O[o] * Yo[o] for o in O) >= LB, 'Wave_LB')
    solver.Add(sum(cont_O[o] * Yo[o] for o in O) <= UB, 'Wave_UB')

    # (R2) Se o pedido está na wave, todos os seus itens são transportados
    for o in O:
        solver.Add(
            sum(X[(i, o, a)] for i in Io[o] for a in A) == cont_O[o] * Yo[o],
            f'Order_{o}'
        )

    # (R3) Capacidade item a item por corredor (e ativa Ya se o corredor for usado)
    for a in A:
        for i in I:
            demanda_ia = [X[(i, o, a)] for o in O if (i, o, a) in X]
            if demanda_ia:  # só cria a restrição se houver variáveis envolvendo (i,a)
                solver.Add(
                    sum(demanda_ia) <= corredores_dict[a][i] * Ya[a],
                    f'Cap_{i}_{a}'
                )

    # ── Objetivo parametrizado ─────────────────────────────────────────────────
    # max  Σ cont_O[o]*Yo[o]  -  p * Σ Ya[a]
    solver.Maximize(
        sum(cont_O[o] * Yo[o] for o in O)
        - p * sum(Ya[a] for a in A)
    )

    if writelp:
        lp = solver.ExportModelAsLpFormat(False)
        with open(f'modelo_p{p:.4f}.lp', 'w') as f:
            f.write(lp)

    status = solver.Solve()
    g_p = solver.Objective().Value() if status == pywraplp.Solver.OPTIMAL else None
    return g_p, Yo, Ya, X, status

## Bisseção Paramétrica

Busca binária no intervalo $[lo, hi]$ até que $hi - lo < \text{TOLERANCIA}$.

A cada iteração:
1. Testa $p_{mid} = (lo + hi) / 2$
2. Se $g(p_{mid}) > 0$ → $lo = p_{mid}$ (há solução melhor que $p_{mid}$)
3. Se $g(p_{mid}) \leq 0$ → $hi = p_{mid}$ (nenhuma solução supera $p_{mid}$)

In [11]:
tempo_inicio = time.time()

lo = 0.0
hi = float(UB)   # limite superior seguro: todos os itens em 1 corredor

melhor_Yo = None
melhor_Ya = None
melhor_X  = None
melhor_metrica = 0.0
historico = []  # registro de cada iteração

print(f"{'Iter':>4}  {'lo':>8}  {'hi':>8}  {'p_mid':>8}  {'g(p)':>10}  {'Largura':>8}  {'Ação':>6}")
print('-' * 65)

for iteracao in range(1, MAX_ITER + 1):
    largura = hi - lo
    if largura < TOLERANCIA:
        print(f'\n✓ Convergiu após {iteracao - 1} iterações (largura = {largura:.6f} < {TOLERANCIA})')
        break

    p_mid = (lo + hi) / 2.0

    g_p, Yo, Ya, X, status = resolve_parametrico(p_mid)

    if status != pywraplp.Solver.OPTIMAL:
        print(f'  [iter {iteracao}] Solver não encontrou solução ótima para p={p_mid:.4f}. Abortando.')
        break

    acao = ''
    if g_p > 0:
        lo = p_mid
        acao = 'lo ↑'
        # Guarda a melhor solução viável encontrada
        melhor_Yo = Yo
        melhor_Ya = Ya
        melhor_X  = X
    else:
        hi = p_mid
        acao = 'hi ↓'

    historico.append({'iter': iteracao, 'lo': lo, 'hi': hi, 'p_mid': p_mid, 'g_p': g_p, 'largura': largura})
    print(f"{iteracao:>4}  {lo:>8.4f}  {hi:>8.4f}  {p_mid:>8.4f}  {g_p:>10.4f}  {largura:>8.4f}  {acao:>6}")

tempo_total = time.time() - tempo_inicio
print(f'\nTempo total: {tempo_total:.1f}s')

Iter        lo        hi     p_mid        g(p)   Largura    Ação
-----------------------------------------------------------------
   1    0.0000   76.5000   76.5000    -78.0000  153.0000    hi ↓
   2    0.0000   38.2500   38.2500     -1.5000   76.5000    hi ↓
   3   19.1250   38.2500   19.1250     45.6250   38.2500    lo ↑
   4   28.6875   38.2500   28.6875     17.6250   19.1250    lo ↑
   5   33.4688   38.2500   33.4688      8.0625    9.5625    lo ↑
   6   35.8594   38.2500   35.8594      3.2812    4.7812    lo ↑
   7   37.0547   38.2500   37.0547      0.8906    2.3906    lo ↑
   8   37.0547   37.6523   37.6523     -0.3047    1.1953    hi ↓
   9   37.3535   37.6523   37.3535      0.2930    0.5977    lo ↑
  10   37.3535   37.5029   37.5029     -0.0059    0.2988    hi ↓
  11   37.4282   37.5029   37.4282      0.1436    0.1494    lo ↑
  12   37.4656   37.5029   37.4656      0.0688    0.0747    lo ↑
  13   37.4843   37.5029   37.4843      0.0315    0.0374    lo ↑
  14   37.4936   37.5029

## Extração e Validação da Solução Final

In [ ]:
if melhor_Yo is not None:
    Pedidos_Escolhidos, Corredores_Utilizados, Transportes = extrai_solucao(melhor_Yo, melhor_Ya, melhor_X)

    sobras = calcula_sobras(Transportes)

    print('=' * 55)
    print(f'  Instância:                {instancia}')
    print(f'  Pedidos no backlog:       {n_o}')
    print(f'  Itens distintos:          {n_i}')
    print(f'  Corredores disponíveis:   {n_a}')
    print(f'  LB / UB:                  {LB} / {UB}')
    print('-' * 55)
    print(f'  Pedidos na wave:          {len(Pedidos_Escolhidos)}')
    print(f'  Corredores usados:        {len(Corredores_Utilizados)}')
    print(f'  Total de unidades:        {int(tot_pedidos(Transportes))}')
    print(f'  Métrica (itens/corredor): {metrica(Transportes, Corredores_Utilizados):.4f}')
    print(f'  Valor ótimo estimado:     {lo:.4f}  (intervalo final: [{lo:.4f}, {hi:.4f}])')
    print(f'  Sobra de capacidade:      {sobras:.1f}  {"✓ OK" if sobras >= -1e-6 else "✗ INVIÁVEL"}')
    print('=' * 55)
    print(f'  Pedidos escolhidos:       {Pedidos_Escolhidos}')
    print(f'  Corredores utilizados:    {Corredores_Utilizados}')
else:
    print('Nenhuma solução viável encontrada.')

## Histórico de Convergência

In [ ]:
df_hist = pd.DataFrame(historico)
display(df_hist.style.format({
    'lo': '{:.4f}', 'hi': '{:.4f}', 'p_mid': '{:.4f}',
    'g_p': '{:.4f}', 'largura': '{:.4f}'
}).bar(subset=['largura'], color='#d4e8f7'))

## Detalhe dos Transportes

In [ ]:
if melhor_X is not None:
    df_transp = pd.DataFrame(
        [(u[0], u[1], u[2], Transportes[u]) for u in Transportes],
        columns=['Item', 'Pedido', 'Corredor', 'Quantidade']
    ).sort_values(by=['Corredor', 'Pedido'])
    display(df_transp)